<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_CMVG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [19]:
# Cell 1: Setup
# ---------------------------------
# This cell imports the necessary libraries for the calculation.

import pandas as pd
import numpy as np

print("Libraries imported successfully.")


# In[2]:
# Cell 2: Step 1 - Initial Portfolio Data
# ---------------------------------------
# This cell defines the initial portfolio data and loads it into a pandas DataFrame.
# The DataFrame is displayed as the output of this cell, mirroring the first step in the report.

data = [
    {'Position ID': 1, 'Bucket': 6, 'Commodity Name': 'UKNBP', 'Commodity Location': 'UK', 'Option Maturity': '1Y', 'Gross Sensitivity': 74755},
    {'Position ID': 2, 'Bucket': 6, 'Commodity Name': 'NLTTF', 'Commodity Location': 'Netherlands', 'Option Maturity': '6M', 'Gross Sensitivity': -82734},
    {'Position ID': 3, 'Bucket': 2, 'Commodity Name': 'Brent', 'Commodity Location': 'North Sea', 'Option Maturity': '1Y', 'Gross Sensitivity': 311994},
    {'Position ID': 4, 'Bucket': 2, 'Commodity Name': 'Brent', 'Commodity Location': 'North Sea', 'Option Maturity': '6M', 'Gross Sensitivity': 1636643},
    {'Position ID': 5, 'Bucket': 2, 'Commodity Name': 'Brent', 'Commodity Location': 'North Sea', 'Option Maturity': '6M', 'Gross Sensitivity': -1271914}
]

portfolio_df = pd.DataFrame(data)

print("--- Step 1: Initial Portfolio ---")
portfolio_df


Libraries imported successfully.
--- Step 1: Initial Portfolio ---


,Position ID,Bucket,Commodity Name,Commodity Location,Option Maturity,Gross Sensitivity
0,1,6,UKNBP,UK,1Y,74755
1,2,6,NLTTF,Netherlands,6M,-82734
2,3,2,Brent,North Sea,1Y,311994
3,4,2,Brent,North Sea,6M,1636643
4,5,2,Brent,North Sea,6M,-1271914


In [20]:
# Cell 3: Step 3 - Calculate Net Sensitivities
# ---------------------------------------------
# As per Article 325f, sensitivities for the same risk factor are netted.
# The resulting DataFrame of net sensitivities is displayed as the output.

print("--- Step 3: Net Sensitivities ---")

net_sensitivities_df = portfolio_df.groupby(['Bucket', 'Commodity Name', 'Commodity Location', 'Option Maturity'])['Gross Sensitivity'].sum().reset_index()
net_sensitivities_df.rename(columns={'Gross Sensitivity': 'Net Sensitivity'}, inplace=True)
net_sensitivities_df['Risk Factor'] = net_sensitivities_df['Commodity Name'] + '_' + net_sensitivities_df['Option Maturity']

net_sensitivities_df[['Bucket', 'Risk Factor', 'Net Sensitivity']]


--- Step 3: Net Sensitivities ---


,Bucket,Risk Factor,Net Sensitivity
0,2,Brent_1Y,311994
1,2,Brent_6M,364729
2,6,NLTTF_6M,-82734
3,6,UKNBP_1Y,74755


In [21]:
# Cell 4: Step 4 - Calculate Weighted Sensitivities
# -------------------------------------------------
# Each net sensitivity is multiplied by its 100% risk weight (Article 325ax).
# The sum of weighted sensitivities (S_b) for each bucket is also calculated.

print("--- Step 4: Weighted Sensitivities ---")

VEGA_RISK_WEIGHT = 1.0
net_sensitivities_df['Weighted Sensitivity'] = net_sensitivities_df['Net Sensitivity'] * VEGA_RISK_WEIGHT

bucket_sums = net_sensitivities_df.groupby('Bucket')['Weighted Sensitivity'].sum()
S_2 = bucket_sums.loc[2]
S_6 = bucket_sums.loc[6]

print(f"Sum for Bucket 2 (S_2): {S_2:,.0f}")
print(f"Sum for Bucket 6 (S_6): {S_6:,.0f}")

net_sensitivities_df[['Bucket', 'Risk Factor', 'Weighted Sensitivity']]

--- Step 4: Weighted Sensitivities ---
Sum for Bucket 2 (S_2): 676,723
Sum for Bucket 6 (S_6): -7,979


,Bucket,Risk Factor,Weighted Sensitivity
0,2,Brent_1Y,311994.0
1,2,Brent_6M,364729.0
2,6,NLTTF_6M,-82734.0
3,6,UKNBP_1Y,74755.0


In [22]:
# Cell 5: Steps 5 & 6 - Define Correlation Parameters (Medium Scenario)
# ---------------------------------------------------------------------
# This cell defines the correlations based on Articles 325at, 325ay, and 325au.

print("--- Steps 5 & 6: Correlation Parameters (Medium Scenario) ---")

rho_delta_2 = 1.00 * 0.99 * 1.00
t_k, t_l = 1.0, 0.5
rho_opt_maturity = np.exp(-0.01 * (abs(t_k - t_l) / min(t_k, t_l)))
rho_2 = rho_delta_2 * rho_opt_maturity

rho_delta_6 = 0.65 * 0.99 * 0.999
rho_6 = rho_delta_6 * rho_opt_maturity

gamma_2_6 = 0.20

medium_params = {'rho_2': rho_2, 'rho_6': rho_6, 'gamma_2_6': gamma_2_6}

medium_corr_df = pd.DataFrame([
    {'Parameter': 'Intra-Bucket Correlation (rho_2)', 'Value': rho_2},
    {'Parameter': 'Intra-Bucket Correlation (rho_6)', 'Value': rho_6},
    {'Parameter': 'Cross-Bucket Correlation (gamma_2_6)', 'Value': gamma_2_6}
])
medium_corr_df.set_index('Parameter')

--- Steps 5 & 6: Correlation Parameters (Medium Scenario) ---


,Value
Parameter,
Intra-Bucket Correlation (rho_2),0.980149
Intra-Bucket Correlation (rho_6),0.636460
Cross-Bucket Correlation (gamma_2_6),0.200000


In [23]:
# Cell 6: Step 7 - Intra-Bucket Aggregation (Medium Scenario)
# -----------------------------------------------------------
# The bucket-specific capital (K_b) is calculated for the medium scenario.

def calculate_k_b(sensitivities, correlation):
    """Calculates the intra-bucket capital (K_b) using a vectorized formula."""
    if len(sensitivities) < 2: return np.abs(sensitivities[0]) if len(sensitivities) > 0 else 0
    sum_sq = np.sum(sensitivities**2)
    sum_sens = np.sum(sensitivities)
    k_b_sq = (1 - correlation) * sum_sq + correlation * (sum_sens**2)
    return np.sqrt(k_b_sq)

ws_b2 = net_sensitivities_df[net_sensitivities_df['Bucket'] == 2]['Weighted Sensitivity'].values
ws_b6 = net_sensitivities_df[net_sensitivities_df['Bucket'] == 6]['Weighted Sensitivity'].values

K2_med = calculate_k_b(ws_b2, medium_params['rho_2'])
K6_med = calculate_k_b(ws_b6, medium_params['rho_6'])

print("--- Step 7: Intra-Bucket Capital (Medium Scenario) ---")
k_b_medium_df = pd.DataFrame([
    {'Bucket': 2, 'K_b (Medium)': K2_med},
    {'Bucket': 6, 'K_b (Medium)': K6_med}
]).set_index('Bucket')
k_b_medium_df.style.format('{:,.0f}')



--- Step 7: Intra-Bucket Capital (Medium Scenario) ---


,K_b (Medium)
Bucket,
2,"673,377"
6,"67,531"


In [24]:
# In[7]:
# Cell 7: Step 8 - Across-Bucket Aggregation (Medium Scenario)
# ------------------------------------------------------------
# The bucket capitals are aggregated to find the total capital for the medium scenario.

print("--- Step 8: Across-Bucket Capital (Medium Scenario) ---")
sum_of_k_squares = K2_med**2 + K6_med**2
cross_bucket_term = 2 * medium_params['gamma_2_6'] * S_2 * S_6
capital_medium = np.sqrt(sum_of_k_squares + cross_bucket_term)

print(f"Medium Scenario Total Capital: {capital_medium:,.0f}")

--- Step 8: Across-Bucket Capital (Medium Scenario) ---
Medium Scenario Total Capital: 675,157


In [25]:
# Cell 8: Step 9 (Part 1) - Derive High and Low Correlation Scenarios
# -------------------------------------------------------------------
# The stressed scenario parameters are derived from the medium scenario baseline.

print("--- Step 9: Derived Scenario Coefficients ---")
high_params = {
    'rho_2': min(rho_2 * 1.25, 1.0), 'rho_6': min(rho_6 * 1.25, 1.0), 'gamma_2_6': min(gamma_2_6 * 1.25, 1.0)
}
low_params = {
    'rho_2': max(2 * rho_2 - 1, 0.75 * rho_2), 'rho_6': max(2 * rho_6 - 1, 0.75 * rho_6), 'gamma_2_6': max(2 * gamma_2_6 - 1, 0.75 * gamma_2_6)
}

derived_params_df = pd.DataFrame([
    {'Scenario': 'Medium', 'rho_2': medium_params['rho_2'], 'rho_6': medium_params['rho_6'], 'gamma_2_6': medium_params['gamma_2_6']},
    {'Scenario': 'High', 'rho_2': high_params['rho_2'], 'rho_6': high_params['rho_6'], 'gamma_2_6': high_params['gamma_2_6']},
    {'Scenario': 'Low', 'rho_2': low_params['rho_2'], 'rho_6': low_params['rho_6'], 'gamma_2_6': low_params['gamma_2_6']}
]).set_index('Scenario')

derived_params_df

--- Step 9: Derived Scenario Coefficients ---


,rho_2,rho_6,gamma_2_6
Scenario,,,
Medium,0.980149,0.636460,0.20
High,1.000000,0.795575,0.25
Low,0.960299,0.477345,0.15


In [26]:
# Cell 9: Step 9 (Part 2) - Calculate All Scenario Capitals
# ---------------------------------------------------------
# The total capital is recalculated for the high and low correlation scenarios.

def calculate_scenario_capital(params, sensitivities_df, bucket_sums):
    """Calculates the final capital charge for a given scenario."""
    ws_b2 = sensitivities_df[sensitivities_df['Bucket'] == 2]['Weighted Sensitivity'].values
    ws_b6 = sensitivities_df[sensitivities_df['Bucket'] == 6]['Weighted Sensitivity'].values
    K_2 = calculate_k_b(ws_b2, params['rho_2'])
    K_6 = calculate_k_b(ws_b6, params['rho_6'])
    S_2 = bucket_sums.loc[2]
    S_6 = bucket_sums.loc[6]
    sum_of_k_squares = K_2**2 + K_6**2
    cross_bucket_term = 2 * params['gamma_2_6'] * S_2 * S_6
    return np.sqrt(sum_of_k_squares + cross_bucket_term)

capital_high = calculate_scenario_capital(high_params, net_sensitivities_df, bucket_sums)
capital_low = calculate_scenario_capital(low_params, net_sensitivities_df, bucket_sums)

print("--- Scenario Capital Results ---")
results_data = [
    {'Scenario': 'Medium', 'Total Capital': capital_medium},
    {'Scenario': 'High', 'Total Capital': capital_high},
    {'Scenario': 'Low', 'Total Capital': capital_low}
]
results_df = pd.DataFrame(results_data).set_index('Scenario')
results_df.style.format('{:,.0f}')

--- Scenario Capital Results ---


,Total Capital
Scenario,
Medium,"675,157"
High,"676,644"
Low,"673,667"


In [27]:
# Cell 10: Step 10 - Final Charge Calculation
# -------------------------------------------
# The final capital requirement is the highest of the three scenario results.

print("--- Step 10: Final Charge Calculation ---")
final_capital_charge = max(capital_medium, capital_high, capital_low)

final_result_df = pd.DataFrame([{'Final Commodity Vega Capital Requirement': final_capital_charge}])
final_result_df.style.format('{:,.0f}')


--- Step 10: Final Charge Calculation ---


,Final Commodity Vega Capital Requirement
0,"676,644"
